In [3]:
import os
import uuid
import hashlib
from datetime import datetime
import numpy as np
import pandas as pd
from cryptography.fernet import Fernet
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer

PATH = "complications.csv"


# Przygotowanie danych

df = pd.read_csv(PATH)

# Wybieramy kluczowe kolumny dla uproszczenia (dane demograficzne, kliniczne i target)
cols = ['ID', 'AGE', 'SEX', 'S_AD_ORIT', 'D_AD_ORIT', 'K_BLOOD', 'L_BLOOD', 'LET_IS']
df = df[cols].copy()

# Imputacja braków danych (medianą)
for col in ['AGE', 'S_AD_ORIT', 'D_AD_ORIT', 'K_BLOOD', 'L_BLOOD']:
    df[col] = df[col].fillna(df[col].median())

# Cel: 0 - przeżycie, 1 - zgon (LET_IS > 0)
df['LET_IS'] = (df['LET_IS'].fillna(0).astype(int) > 0).astype(int)
df["diagnosis"] = df["LET_IS"].map({1: "Lethal", 0: "Alive"})


# Szyfrowanie i integralność danych

key = Fernet.generate_key()
cipher = Fernet(key)
print("Klucz szyfrujący wygenerowany.")

# Szyfrowanie kolumny diagnozy
df["diagnosis_enc"] = df["diagnosis"].apply(
    lambda x: cipher.encrypt(x.encode("utf-8")).decode("utf-8")
)
print("\nPrzykład zaszyfrowanej diagnozy:")
print(df[["diagnosis", "diagnosis_enc"]].head())

# Funkcja do haszowania pliku
def compute_file_hash(path: str, algo: str = "sha256") -> str:
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            h.update(chunk)
    return h.hexdigest()

# Zapisanie wyczyszczonej wersji do nowego pliku
SECURE_PATH = "complications_secure.csv"
df.to_csv(SECURE_PATH, index=False)
original_hash = compute_file_hash(SECURE_PATH, "sha256")
print(f"\nSHA-256 pliku danych: {original_hash}")

Klucz szyfrujący wygenerowany.

Przykład zaszyfrowanej diagnozy:
  diagnosis                                      diagnosis_enc
0     Alive  gAAAAABp5hdDDWDvRNa92aQ38NLtW2hFVF6QxBWWenLdOx...
1     Alive  gAAAAABp5hdDy8ZOUm4zmWTN83z-cN7iu82tzxHNAY7us0...
2     Alive  gAAAAABp5hdDBSqsYEqHyTGn4GLO3IInR_65PY3iiXV774...
3     Alive  gAAAAABp5hdDFjFxDb3MeDb-LT1r3jitLQugxeKvz7wyex...
4     Alive  gAAAAABp5hdDgwJSRoPVdfghR9rQNrQlRbn6GwMGy273Ky...

SHA-256 pliku danych: 9f1faacfac09683e7f3553c1038b4582b164d8bab6cc90a30c78f265a5e85b8d


In [4]:

# Pseudonimizacja i anonimizacja


# 3.1 Pseudonimizacja
df["patient_id"] = [str(uuid.uuid4()) for _ in range(len(df))]
df = df.drop(columns=["ID"]) # Usuwamy oryginalne ID

# 3.2 Anonimizacja (generalizacja wieku)
def age_to_group(age):
    if age < 40: return "<40"
    elif age < 60: return "40-59"
    elif age < 80: return "60-79"
    else: return ">=80"

df["age_group"] = df["AGE"].apply(age_to_group)
df_anonym = df.drop(columns=["AGE"]) # Usuwamy dokładny wiek dla zachowania prywatności

print("\nZanonimizowane dane (pierwsze wiersze):")
print(df_anonym[["patient_id", "age_group", "SEX", "S_AD_ORIT", "diagnosis"]].head())


# Kontrola dostępu i audyt

ROLES = {
    "admin": {
        "columns": df_anonym.columns.tolist()  # Pełny dostęp do wszystkiego
    },
    "doctor": {
        # Widzi dane kliniczne i jawną diagnozę, nie widzi hashy
        "columns": ["patient_id", "age_group", "SEX", "S_AD_ORIT", "D_AD_ORIT", "K_BLOOD", "L_BLOOD", "LET_IS", "diagnosis"]
    },
    "analyst": {
        # Analityk widzi tylko dane numeryczne i grupy wiekowe (brak diagnozy)
        "columns": ["age_group", "SEX", "S_AD_ORIT", "D_AD_ORIT", "K_BLOOD", "L_BLOOD", "LET_IS"]
    }
}

audit_log = []

def audit(user: str, role: str, action: str, status: str):
    audit_log.append({
        "time": datetime.now().isoformat(timespec="seconds"),
        "user": user, "role": role, "action": action, "status": status
    })

def get_data_view(df_in: pd.DataFrame, role: str, user: str = "unknown") -> pd.DataFrame:
    if role not in ROLES:
        audit(user, role, "ACCESS_DENIED_UNKNOWN_ROLE", "denied")
        raise ValueError(f"Nieznana rola: {role}")

    allowed_cols = [c for c in ROLES[role]["columns"] if c in df_in.columns]
    view = df_in[allowed_cols].copy()
    audit(user, role, f"ACCESS_GRANTED_{len(allowed_cols)}_columns", "granted")
    return view

# Testujemy dostęp
df_doctor_view = get_data_view(df_anonym, role="doctor", user="dr_kowalski")
df_analyst_view = get_data_view(df_anonym, role="analyst", user="analityk_nowak")

print("\n[Etap 4] Widok lekarza (kolumny):", df_doctor_view.columns.tolist())
print("Widok analityka (kolumny):", df_analyst_view.columns.tolist())
print("Ostatni log audytu:", audit_log[-1])


Zanonimizowane dane (pierwsze wiersze):
                             patient_id age_group  SEX  S_AD_ORIT diagnosis
0  5c9c7d77-3b61-4455-95e5-2d97883d7b03     60-79    1      180.0     Alive
1  5f905cea-8aee-4cfc-bf56-6f5fe305ff13     40-59    1      120.0     Alive
2  e174284b-0a47-4e72-8389-1ff03e19f05e     40-59    1      180.0     Alive
3  7f3c2724-18b9-424a-826c-3181fcd9b62d     60-79    0      120.0     Alive
4  b3f6b908-0200-473b-bcce-2a537a08db84     60-79    1      160.0     Alive

[Etap 4] Widok lekarza (kolumny): ['patient_id', 'age_group', 'SEX', 'S_AD_ORIT', 'D_AD_ORIT', 'K_BLOOD', 'L_BLOOD', 'LET_IS', 'diagnosis']
Widok analityka (kolumny): ['age_group', 'SEX', 'S_AD_ORIT', 'D_AD_ORIT', 'K_BLOOD', 'L_BLOOD', 'LET_IS']
Ostatni log audytu: {'time': '2026-04-20T14:12:13', 'user': 'analityk_nowak', 'role': 'analyst', 'action': 'ACCESS_GRANTED_7_columns', 'status': 'granted'}
